# 🔍 Data Quality Assurance Dashboard\n

## Comprehensive Data Quality Analysis\n

This notebook provides a complete data quality assessment for our YouTube analytics platform.

In [ ]:
# 🔍 Data Quality Assessment Setup\n
import sys\n
import os\n
sys.path.append(os.path.abspath('.'))\n
sys.path.append(os.path.abspath('../..'))\n
\n
import pandas as pd\n
import numpy as np\n
from datetime import datetime, timedelta\n
import warnings\n
warnings.filterwarnings('ignore')\n
\n
# Import our custom modules\n
from src.youtubeviz.data import load_youtube_data\n
from src.youtubeviz.utils import safe_head\n
\n
print('✅ All imports successful!')

In [ ]:
# 📊 Load and examine data\n
print('📊 Loading YouTube data...')\n
df = load_youtube_data()\n
print(f'📈 Dataset loaded: {len(df):,} records')\n
print(f'🎤 Artists: {df["artist_name"].nunique()}')\n
print(f'🎵 Videos: {df["video_id"].nunique()}')\n
\n
# Show data overview\n
print('\n📋 Data Overview:')\n
safe_head(df, 3)

In [ ]:
# 🎯 Compute KPIs function\n
def compute_kpis(df):\n
    \"\"\"Compute key performance indicators for data quality.\"\"\"\n
    \n
    kpis = {}\n
    \n
    # Basic metrics\n
    kpis['total_records'] = len(df)\n
    kpis['unique_videos'] = df['video_id'].nunique()\n
    kpis['unique_artists'] = df['artist_name'].nunique()\n
    \n
    # Data quality metrics\n
    kpis['completeness_score'] = (1 - df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100\n
    kpis['duplicate_rate'] = (len(df) - len(df.drop_duplicates())) / len(df) * 100\n
    \n
    # Artist distribution\n
    artist_counts = df['artist_name'].value_counts()\n
    kpis['artist_distribution'] = artist_counts.to_dict()\n
    \n
    # Date range\n
    if 'metrics_date' in df.columns:\n
        kpis['date_range'] = {\n
            'start': df['metrics_date'].min(),\n
            'end': df['metrics_date'].max()\n
        }\n
    \n
    return kpis\n
\n
# Compute KPIs\n
kpis = compute_kpis(df)\n
print('✅ KPIs computed successfully')\n
kpis

In [ ]:
# 🏆 Overall Data Quality Score\n
def calculate_overall_quality_score(kpis):\n
    \"\"\"Calculate overall data quality score.\"\"\"\n
    \n
    # Completeness (40% weight)\n
    completeness_score = kpis.get('completeness_score', 0) * 0.4\n
    \n
    # Uniqueness (30% weight) - lower duplicate rate is better\n
    uniqueness_score = max(0, 100 - kpis.get('duplicate_rate', 0)) * 0.3\n
    \n
    # Artist balance (20% weight) - more balanced is better\n
    artist_counts = list(kpis.get('artist_distribution', {}).values())\n
    if artist_counts:\n
        cv = np.std(artist_counts) / np.mean(artist_counts) if np.mean(artist_counts) > 0 else 1\n
        balance_score = max(0, 100 - cv * 50) * 0.2\n
    else:\n
        balance_score = 0\n
    \n
    # Recency (10% weight)\n
    recency_score = 100 * 0.1  # Assume recent for now\n
    \n
    total_score = completeness_score + uniqueness_score + balance_score + recency_score\n
    return min(100, max(0, total_score))\n
\n
quality_score = calculate_overall_quality_score(kpis)\n
print(f'🏆 OVERALL DATA QUALITY SCORE: {quality_score:.1f}%')\n
\n
if quality_score >= 95:\n
    print('🟢 EXCELLENT - Data quality is outstanding')\n
elif quality_score >= 85:\n
    print('🟡 GOOD - Data quality is acceptable with minor issues')\n
elif quality_score >= 70:\n
    print('🟠 FAIR - Data quality needs improvement')\n
else:\n
    print('🔴 POOR - Data quality requires immediate attention')

In [ ]:
# 📊 Artist Performance Summary\n
print('🎤 Artist Performance Summary:')\n
print('=' * 50)\n
\n
for artist, count in kpis['artist_distribution'].items():\n
    percentage = (count / kpis['total_records']) * 100\n
    print(f'🎵 {artist}: {count:,} records ({percentage:.1f}%)')\n
\n
print(f'\n📊 Total Records: {kpis["total_records"]:,}')\n
print(f'🎵 Unique Videos: {kpis["unique_videos"]:,}')\n
print(f'🎤 Unique Artists: {kpis["unique_artists"]}')\n
print(f'📈 Completeness: {kpis["completeness_score"]:.1f}%')\n
print(f'🔄 Duplicate Rate: {kpis["duplicate_rate"]:.1f}%')